In [45]:
### IMPORTAÇÃO DE BIBLIOTECAS ####

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from pathlib import Path
import time
import datetime as dt
import warnings
import unicodedata

warnings.filterwarnings('ignore',category=FutureWarning)
user = "wconceicao"
password = "zJm7$j%qRU@WoCxM"
host = "dw-ro.data.grupoa.education"
port = "5432"
database = "postgres"
password_encoded = quote_plus(password)
engine = create_engine(
    f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{database}"
    )

In [46]:
    ### FUNÇÕES E DEPARA #### 

def padrao_e_filtro(
            df,
            mapa_colunas,
            df_sispag,
            df_rod_atual=None,
            area=None,
            type=None,
            program=None,
            df_blacklist=None,
            leadScoreOlos = None,
            leadScoreBlip = None,
            qtdcalls=None,
            limite = 10,
            olos_ultimo_contato=None,
            blip_ultimo_contato=None        
    ):
        try:
            df = padronizar(df,mapa_colunas)
            df = dedup_basica(df,key='copy')
            df = filtrar_base(df,area=area,program=program,type=type)
            df = limp_min(df)
            df = padrao_telefone(df)
            df = val_n_movel(df)
            if df_rod_atual is not None:
                df = rod_atual(df,df_rod_atualmente=df_rod_atual)
            if df_blacklist is not None:
                df = aplicar_blacklist(df,df_blacklist)
            if leadScoreOlos is not None:
                df = score_olos(df,leadScoreOlos)    
            if leadScoreBlip is not None:
                df = score_blip(df,leadScoreBlip)  
            if qtdcalls is not None:
                df = qtdCalls(df,qtdcalls,limite=limite)  
            df = remover_duplicadas(df,key='copy')
            df = ultima_compra(df,df_sispag)
            df = last_conct_olos(df,olos_ultimo_contato=olos_ultimo_contato)
            df = last_conct_blip(df,blip_ultimo_contato=blip_ultimo_contato)
            df = filtro_final(df)
            
            return df.reset_index(drop=True)
        except Exception as e:
            return f'Erro ao processar: {e}'

DePara_colunas = {
        'base_inativa':{
            'email':'email',
            'phone_1':'phone',
            'product':'program',
            'area':'area',
            'copy':'copy',
            'type':'type'
        },

        'base_ativa':{
            'email':'email',
            'phone':'phone',
            'program':'program',
            'area':'area',
            'copy':'copy'
        },

        'base_carrinho':{
            'Area':'area',
            'email':'email',
            'celular':'phone',
            'programa':'program',
            'copy':'copy'
        },
        
        'base_hubspot':{
        }   
    }

def executar_qry(query,nome_df):

        inicio = time.time()

        try:
            print(f'Executando: {nome_df}...')
            df = pd.read_sql(text(query),engine)

            tempo_total = time.time() - inicio
            minutos = int(tempo_total // 60)
            segundos = tempo_total % 60

            if minutos > 0:
                tempo = f'{minutos}min {segundos:.1f}s'
            else:
                tempo = f'{segundos:.2f}s' 

            num_linhas = len(df)

            print(f'✅ {nome_df}: {num_linhas} linhas | ⏱️{tempo}')

            return df       

        except Exception as e:
            tempo_total = time.time() - inicio
            print(f' Erro em {nome_df} após {tempo_total:.2f}s: {e}')
            return None
        
def filtrar_base(df,area=None,program=None,type=None):
        resultado = df.copy()

        if area is not None:

            areas = area if isinstance(area, list) else [area]
            resultado = resultado[
                resultado['area']
                .str.upper()
                .isin([a.upper() for a in areas])
            ]

        if type is not None:

            types = area if isinstance(type, list) else [type]
            resultado = resultado[
                resultado['type']
                .str.upper()
                .isin([t.upper() for t in types])
            ]

        if program is not None:
            resultado = resultado[
                resultado['program'] 
                .str.contains(program,case=False,na=False)
            ]

        return resultado.reset_index(drop=True)        

def padronizar(df,mapa_colunas):
        df_padrao = df.rename(columns=mapa_colunas)
        return df_padrao[list(mapa_colunas.values())]

def limp_min(df):
        resultado = df.copy()

        resultado = resultado[
            resultado['email'].notna() &
            resultado['phone'].notna()
        ]

        resultado = resultado[
            (resultado['email'].str.strip() != '') &
            (resultado['phone'].str.strip() != '')
        ]

        resultado['copy'] = resultado['email']+';'+resultado['phone']

        return resultado.reset_index(drop=True)

def dedup_basica(df,key='copy'):
        return(
            df
            .drop_duplicates(subset=key,keep='first')
            .reset_index(drop=True)
        )

def remover_duplicadas(df,key='copy'):
        return(
            df
            .sort_values(by=key)
            .drop_duplicates(subset=key,keep='first')
            .reset_index(drop=True)
        )

def padrao_telefone(df):
        resultado = df.copy()

        resultado['email'] = (
            resultado['email'].str.strip().str.lower()
        )
        
        resultado['phone'] = (
            resultado['phone'].str.replace(r'\D','',regex=True)
        )

        resultado['copy'] = resultado['email'] + ';' + resultado['phone']

        return resultado.reset_index(drop=True)

def val_n_movel(df):

        resultado = df.copy()

        resultado['is_movel'] = (
            resultado['phone'].str.len().eq(11) & 
            resultado['phone'].str[2].eq('9')
        )

        return resultado

def aplicar_blacklist (df,df_blacklist):
        resultado = df.copy()
        blacklist = df_blacklist.copy()
        
        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )
        
        blacklist['telefone'] = (
            blacklist['telefone'].astype(str).str.replace(r'\D','',regex=True)
        )
        
        resultado['Na_blacklist'] = resultado['phone'].isin(df_blacklist['telefone'])
        return resultado

def rod_atual(df,df_rod_atualmente):
        resultado = df.copy()
        df_rod_atual = df_rod_atualmente.copy()

        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )

        resultado['rod_atual'] = resultado['phone'].isin(df_rod_atual['fone_1'])

        return resultado

def score_olos(df,ls_olos):
        resultado = df.copy()
        leadscore_olos = ls_olos.copy()

        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )

        leadscore_olos['phone_number'] = (
            leadscore_olos['phone_number'].astype(str).str.replace(r'\D','',regex=True)
        )

        resultado['No_leadScore_Olos'] = resultado['phone'].isin(leadscore_olos['phone_number'])

        return resultado

def score_blip(df,ls_blip):
        resultado = df.copy()
        leadscore_blip = ls_blip.copy()

        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )

        leadscore_blip['phone_number'] = (
            leadscore_blip['phone_number'].astype(str).str.replace(r'\D','',regex=True)
        )

        resultado['No_leadScore_blip'] = resultado['phone'].isin(leadscore_blip['phone_number'])

        return resultado

def qtdCalls(df,df_qtd,limite=10):
        resultado = df.copy()
        qtd = df_qtd.copy()

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        qtd['phone_number'] = qtd['phone_number'].astype(str).str.replace(r'\D','',regex=True)

        resultado = resultado.merge(
            qtd[['phone_number','TOTAL_CALLS']],
            left_on='phone',
            right_on='phone_number',
            how='left'
        )

        resultado['acima_limite'] = (
            resultado['TOTAL_CALLS'].fillna(0).astype(int) > limite
        )

        resultado = resultado.drop(columns=['phone_number','TOTAL_CALLS'])
        
        return resultado

def ultima_compra(df,df_sispag):
        resultado = df.copy()
        sispag = df_sispag.copy()

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        sispag['celular'] = sispag['celular'].astype(str).str.replace(r'\D','',regex=True)

        resultado['ultima_compra'] = resultado['phone'].isin(sispag['celular'])

        return resultado 

def last_conct_olos(df,olos_ultimo_contato):
        resultado = df.copy()
        tab_olos = olos_ultimo_contato.copy()

        tab_olos['data_olos'] = pd.to_datetime(tab_olos['data_olos'],errors='coerce')
        tab_olos['dias_desde_tab'] = (pd.Timestamp.today() - tab_olos['data_olos']).dt.days
        tab_olos['dias_desde_tab'] = tab_olos['dias_desde_tab'].fillna(0)
        tab_olos['last_conct_olos'] = (tab_olos['dias_desde_tab'] > tab_olos['retorno_em_dias']) & (tab_olos['status_retorno'] == 'retornar')

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        tab_olos['phone'] = tab_olos['phone'].astype(str).str.replace(r'\D','',regex=True).str[-11:]

        resultado = resultado.merge(tab_olos[['phone','last_conct_olos']],left_on='phone',right_on='phone',how='left')
        resultado['last_conct_olos'] = resultado['last_conct_olos'].fillna(True)

        return resultado

def last_conct_blip(df,blip_ultimo_contato):
        resultado = df.copy(deep=True)
        tab_blip = blip_ultimo_contato.copy(deep=True)
        
        tab_blip['data_blip'] = pd.to_datetime(tab_blip['data_blip'],errors='coerce')
        tab_blip['dias_desde_tab'] = (pd.Timestamp.today() - tab_blip['data_blip']).dt.days
        tab_blip['dias_desde_tab'] = tab_blip['dias_desde_tab'].fillna(0)
        tab_blip['last_conct_blip'] = (tab_blip['dias_desde_tab'] > tab_blip['retorno_em_dias']) & (tab_blip['status_retorno'] == 'retornar')

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        tab_blip['contact_id_trimmed'] = tab_blip['contact_id_trimmed'].astype(str).str.replace(r'\D','',regex=True)

        resultado = resultado.merge(tab_blip[
            ['contact_id_trimmed','last_conct_blip']],left_on='phone',right_on='contact_id_trimmed',how='left')
        
        resultado['last_conct_blip'] = resultado['last_conct_blip'].fillna(True)

        return resultado

def exportar_bases(bases_list):
        bases = bases_list.copy()
        pasta_saida = Path(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\bases a finalizar')
        pasta_saida.mkdir(parents=True, exist_ok=True)

        try:
            for nome_base, df_base in bases.items():
                print(f'Base: {nome_base} | Tamanho: {len(df_base)}')
                print(f'Exportando...')
                caminho = pasta_saida/f'{nome_base}.xlsx'
                df_base.to_excel(caminho,index=False)
                print(f'base exportada com sucesso.\n')
        except Exception as e:
            print(f'Erro ao exportar a base {nome_base}')

def filtro_final(df):

        resultado = df.copy()
        resultado['manter_contato'] = (
        resultado['is_movel'] &
        ~resultado.get('Na_blacklist',False) &
        ~resultado.get('No_leadScore_Olos',False) &
        ~resultado.get('No_leadScore_blip',False) &
        ~resultado.get('acima_limite',False) &
        ~resultado.get('ultima_compra',False) & 
        ~resultado.get('rod_atual',False) &
        resultado['last_conct_olos'] &
        resultado['last_conct_blip']
        )
        resultado = resultado[resultado['manter_contato']].reset_index(drop=True) 

        resultado = resultado.drop(
            columns=['is_movel',
                    'Na_blacklist',
                    'No_leadScore_Olos',
                    'No_leadScore_blip',
                    'acima_limite',
                    'ultima_compra',
                    'last_conct_olos',
                    'last_conct_blip',
                    'manter_contato',
                    'rod_atual',
                    'contact_id_trimmed'
                    ])
        return resultado


In [47]:
### QUERY SQL #####

qry_sispag = """
    select
case
	when tipoproduto = 'C' then 'ARTMED360' else 'SECAD'
end as "nome_ies",
	right(replace(replace(replace(replace(app_sispag_pagamento.vd_cliente.celular,'(',''),')',''),'-',''),' ',''),11) "celular",
	vd_produto.tipoproduto::text,
	case
		when vd_compra."codVendedor" is null then date(timezone('America/Sao_Paulo'::text,
		timezone('UTC'::text,
		vd_compra.datahora)))
		when vd_compra."codVendedor"::text = '8000'::text then date(timezone('America/Sao_Paulo'::text,
		timezone('UTC'::text,
		vd_compra.datahora)))
		else date(vd_compra.datahora)
	end as dt,
	vd_compra.compraid as id,
	vd_compra.clienteid as user_id,
	vd_cliente.nome as name,
	lower(vd_cliente.email::text) as email,
	vd_cliente.cidade as city,
	vd_cliente."UF" as state,
	vd_compraitem.produtoid as product_id,
	case
		when vd_produto.nomeresumido::text = 'PROENDÓCRINO'::text then 'PROENDOCRINO'::character varying
		when vd_produto.nomeresumido::text = 'PROENF-URG'::text then 'PROENF/URG'::character varying
		when vd_produto.nomeresumido::text = 'PROFISIO/NEURO'::text then 'PROFISIO/NEF'::character varying
		when vd_produto.nomeresumido::text = 'PROFISIO/TRAUMA'::text then 'PROFISIO/TO'::character varying
		when vd_produto.nomeresumido::text = 'PROMEVET'::text then 'PROMEVET/PA'::character varying
		when vd_produto.nomeresumido::text = 'PROTERAPÊUTICA'::text then 'PROTERAPEUTICA'::character varying
		when vd_produto.nomeresumido::text = 'PROURGEM'::text then 'PROURGEN'::character varying
		when vd_produto.nomeresumido::text = 'PRO-UROLOGIA'::text then 'PROUROLOGIA'::character varying
		else vd_produto.nomeresumido
	end as program,
	case
		when programs.area is not null then programs.area
		else
            case
			when vd_produto.nomeresumido::text = any (array['PROURGEM'::character varying::text,
			'PROENDÓCRINO'::character varying::text,
			'PRO-UROLOGIA'::character varying::text,
			'PROTERAPÊUTICA'::character varying::text]) then 'Medicina'::text
			when vd_produto.nomeresumido::text = 'PROMEVET'::text then 'Veterinária'::text
			when vd_produto.nomeresumido::text = 'PROENF-URG'::text then 'Enfermagem'::text
			when vd_produto.nomeresumido::text = any (array['PROFISIO/NEURO'::character varying::text,
			'PROFISIO/TRAUMA'::character varying::text]) then 'Fisioterapia'::text
			else null::text
		end
	end as area,
	vd_compraitem.valor * vd_compra.valortotal / sum(vd_compraitem.valor) over (partition by vd_compra.compraid) as value,
	vd_compra.formapagamento as payment_type,
	vd_compra."codVendedor" as channel_id,
	case
		when vd_compra."codVendedor" is null then 'eCommerce'::text
		when "left"(vd_compra."codVendedor"::text,
		1) = '8'::text then 'Representantes'::text
		when "left"(vd_compra."codVendedor"::text,
		2) = any (array['90'::text,
		'93'::text]) then 'Receptivo'::text
		when "left"(vd_compra."codVendedor"::text,
		1) = 'R'::text then 'Renovacao'::text
		else 'Call Center'::text
	end as channel
	--vd_compra."valortotal" as valor_compra
from
	app_sispag_pagamento.vd_compra
left join app_sispag_pagamento.vd_compraitem on
	vd_compra.compraid = vd_compraitem.compraid
left join app_sispag_pagamento.vd_produto on
	vd_compraitem.produtoid = vd_produto.produtoid
left join app_sispag_pagamento.vd_request on
	vd_compra.requestid = vd_request.requestid
left join app_sispag_pagamento.vd_cliente on
	vd_compra.clienteid = vd_cliente.clienteid
left join bu_secad.programs on
	vd_produto.nomeresumido::text = programs.program
where
	date(timezone('America/Sao_Paulo'::text,
	timezone('UTC'::text,
	vd_compra.datahora))) >= current_date - interval '3 months'
	and vd_produto.tipoproduto::text in ('P', 'C')
	and vd_request.ambiente::text = 'P'::text
	and (vd_compra."codVendedor"::text <> '123'::text
		or vd_compra."codVendedor" is null)
	and lower(vd_cliente.nome::text) !~~ '%teste%'::text


"""

qtd_calls = """

SELECT  phone_number, COUNT(*) AS "TOTAL_CALLS"

FROM integration_operations.vw_call_center_calls

WHERE campaign_id IN ('1025', '1299', '1553', '1605')

AND start_agent_date >= current_date - interval '2 months'

GROUP BY disposition_nivel_1, phone_number

"""

tab_olos = """

WITH ranked_calls AS (
    SELECT
        right(phone_number,11) as phone,
        customer_id,
        disposition_id,
        disposition_nivel_1,
        TO_CHAR(start_agent_date,'dd/mm/yyyy') as data_olos,
        ROW_NUMBER() OVER (
            PARTITION BY phone_number
            ORDER BY start_agent_date DESC
        ) AS rn,
        CASE 
            WHEN disposition_nivel_1 ~* 'agendado|outra plataforma' THEN 0
            WHEN disposition_nivel_1 ~* 'nao localizado|bolsa gratuita' THEN 15
            WHEN disposition_nivel_1 ~* 'matricula|financeiro|outra formacao|curso de interesse|outra ies|analise de disciplinas|metodologia' THEN 30
            WHEN disposition_nivel_1 ~* 'ligacao falhando|ligacao caiu|sem audio|nao informou|nao atende|apresentacao|metodo pagamento|intresse em evento' THEN 7
            WHEN disposition_nivel_1 ~* 'inadimplente' THEN 60
            WHEN disposition_nivel_1 ~* 'semestre|nao quer EAD|sem tempo|sem interesse' THEN 180
            ELSE 0
        END AS retorno_em_dias,
        CASE 
            WHEN disposition_nivel_1 ~* 'badnumber|fora do publico|nao acionar|desconhece|numero errado|sem afinidade|sem interesse' THEN 'Não retornar'
            ELSE 'retornar'
        END AS status_retorno    
        
    FROM integration_operations.vw_call_center_calls
    WHERE campaign_id IN (1025,1553,1605,1690)
    and start_agent_date >= current_date - interval '6 months'
    and extract(epoch from wrap_duration) > 0
)
SELECT
    phone,customer_id, disposition_nivel_1,data_olos,retorno_em_dias,status_retorno
FROM ranked_calls
WHERE rn = 1

"""

tab_blip = """

with tab_blip as (
	SELECT 
		account_id,
		SUBSTRING(contact_id FROM 3 FOR 11) AS contact_id_trimmed,
		attendance_started_at::date as data_blip,
        ROW_NUMBER() OVER(PARTITION BY contact_id ORDER BY attendance_started_at desc) as rn,
		"template",
		tags,
		CASE 
		    WHEN tags = '' THEN 0
            WHEN tags ~* 'final de semana|transferencia|parou de interagir|abandono|tag|' THEN 0
		    WHEN tags ~* 'automatica|sem tag|metodo de pagamento|evento|bolsa gratuita' THEN 15
		    WHEN tags ~* 'semestre|sem interesse|sem tempo|analise de disciplinas' THEN 180
		    WHEN tags ~* 'matricula|aluno|financeiro|curso|metodologia|outra IES' THEN 30
            WHEN tags ~* 'parou de responder|negociacao' THEN 7
		    ELSE 10
		END AS retorno_em_dias,
		CASE
		    WHEN tags ~* 'numero errado|nao acionar|não relacionado|rel|fora do publico|engano' THEN 'Não_Retorno'
		    ELSE 'retornar'
		END AS status_retorno    
	FROM 
		integration_operations.blip_details_conversations
	WHERE 
		account_id ~* 'secad'
		AND attendance_started_at >= current_date - interval '6 months'
)

select contact_id_trimmed,data_blip,tags,retorno_em_dias,status_retorno
from tab_blip
where rn = 1
order by data_blip desc

"""

lead_score_olos = """

WITH cte_base AS (

  SELECT
      phone_number,
      disposition_id,
      disposition_nivel_1,
      end_agent_date,
      tablename,
      row_number() OVER (PARTITION BY phone_number ORDER BY start_agent_date DESC) AS rn
  FROM integration_operations.vw_call_center_calls
  WHERE 
      campaign_id IN (1025, 1553, 1605, 1690)
      AND start_agent_date >= CURRENT_DATE - INTERVAL '6 months'

),
cte_tags AS (
  SELECT
    phone_number,
    MAX(CASE WHEN rn = 1 THEN disposition_nivel_1 END) AS tag1,
    MAX(CASE WHEN rn = 1 THEN end_agent_date END) AS data_tag1,
    MAX(CASE WHEN rn = 2 THEN disposition_nivel_1 END) AS tag2,
    MAX(CASE WHEN rn = 2 THEN end_agent_date END) AS data_tag2,
    MAX(CASE WHEN rn = 3 THEN disposition_nivel_1 END) AS tag3,
    MAX(CASE WHEN rn = 3 THEN end_agent_date END) AS data_tag3,
    MAX(CASE WHEN rn = 4 THEN disposition_nivel_1 END) AS tag4,
    MAX(CASE WHEN rn = 4 THEN end_agent_date END) AS data_tag4,
    MAX(CASE WHEN rn = 5 THEN disposition_nivel_1 END) AS tag5,
    MAX(CASE WHEN rn = 5 THEN end_agent_date END) AS data_tag5,
    MAX(CASE WHEN rn = 6 THEN disposition_nivel_1 END) AS tag6,
    MAX(CASE WHEN rn = 6 THEN end_agent_date END) AS data_tag6,
    MAX(CASE WHEN rn = 7 THEN disposition_nivel_1 END) AS tag7,
    MAX(CASE WHEN rn = 7 THEN end_agent_date END) AS data_tag7,
    MAX(CASE WHEN rn = 8 THEN disposition_nivel_1 END) AS tag8,
    MAX(CASE WHEN rn = 8 THEN end_agent_date END) AS data_tag8,
    MAX(CASE WHEN rn = 9 THEN disposition_nivel_1 END) AS tag9,
    MAX(CASE WHEN rn = 9 THEN end_agent_date END) AS data_tag9,
    MAX(CASE WHEN rn = 10 THEN disposition_nivel_1 END) AS tag10,
    MAX(CASE WHEN rn = 10 THEN end_agent_date END) AS data_tag10
  FROM 
    cte_base
  GROUP BY 
    phone_number
),cte2 as(
    SELECT *,
    (
        select count(*)
        from unnest(array[tag1,tag2,tag3,tag4,tag5,tag6,tag7,tag8,tag9,tag10]) as x(tag)
        where tag ~* 'BadNumber|Badline_qtd|CancelDial|NoAnswer'
    ) as fail_qtd,
    
    exists(
        select 1 
        from unnest (array[tag1,tag2,tag3,tag4,tag5,tag6,tag7,tag8,tag9,tag10]) as x(tag)
        where tag ~* 'acionar|IMPRODUTIVO NAO ACIONAR|IMPRODUTIVO CAIXA POSTAL NAO ATENDE|IMPRODUTIVO RESPONDEU HSM SEM INTERESSE|HSM|SEM AFINIDADE|desconhece|IMPRODUTIVO Numero Errado|AFINIDADE|recusa'
    ) as nao_acionar
    
    FROM cte_tags
) 

select
    phone_number
    --fail_qtd,
    --nao_acionar
from cte2 


WHERE 
    
    nao_acionar = true OR
    fail_qtd >= 7
    
order by data_tag1 DESC
"""

lead_score_blip = """

WITH blip_bases AS (
    SELECT 
        RIGHT(phonenumber_clean, 11) AS phone_number,
        tag,
        closed_date,
        ROW_NUMBER() OVER (
            PARTITION BY RIGHT(phonenumber_clean,11) 
            ORDER BY started_date DESC
        ) AS rn
    FROM mart_operations.ft_blip_tickets
    WHERE ticket_router IN ('secadativo', 'secadativo2', 'secadreceptivo')
      AND started_date >= current_date - INTERVAL '6 months'
),

cte_tags AS (
    SELECT 
        phone_number,
        MAX(CASE WHEN rn = 1 THEN tag END)  AS tag1,
        MAX(CASE WHEN rn = 1 THEN closed_date END) AS data_tag1,
        MAX(CASE WHEN rn = 2 THEN tag END)  AS tag2,
        MAX(CASE WHEN rn = 2 THEN closed_date END) AS data_tag2,
        MAX(CASE WHEN rn = 3 THEN tag END)  AS tag3,
        MAX(CASE WHEN rn = 3 THEN closed_date END) AS data_tag3,
        MAX(CASE WHEN rn = 4 THEN tag END)  AS tag4,
        MAX(CASE WHEN rn = 4 THEN closed_date END) AS data_tag4,
        MAX(CASE WHEN rn = 5 THEN tag END)  AS tag5,
        MAX(CASE WHEN rn = 5 THEN closed_date END) AS data_tag5,
        MAX(CASE WHEN rn = 6 THEN tag END)  AS tag6,
        MAX(CASE WHEN rn = 6 THEN closed_date END) AS data_tag6,
        MAX(CASE WHEN rn = 7 THEN tag END)  AS tag7,
        MAX(CASE WHEN rn = 7 THEN closed_date END) AS data_tag7,
        MAX(CASE WHEN rn = 8 THEN tag END)  AS tag8,
        MAX(CASE WHEN rn = 8 THEN closed_date END) AS data_tag8,
        MAX(CASE WHEN rn = 9 THEN tag END)  AS tag9,
        MAX(CASE WHEN rn = 9 THEN closed_date END) AS data_tag9,
        MAX(CASE WHEN rn = 10 THEN tag END) AS tag10,
        MAX(CASE WHEN rn = 10 THEN closed_date END) AS data_tag10
    FROM blip_bases
    GROUP BY phone_number
),

cte2 AS (
    SELECT *,
        (
            SELECT COUNT(*)
            FROM unnest(ARRAY[tag1,tag2,tag3,tag4,tag5,tag6,tag7,tag8,tag9,tag10]) AS x(tag)
            WHERE tag ~* 'automatica|relacionado|cae'
        ) AS fail,

        EXISTS (
            SELECT 1
            FROM unnest(ARRAY[tag1,tag2,tag3,tag4,tag5,tag6,tag7,tag8,tag9,tag10]) AS x(tag)
            WHERE tag ~* 'recusa|acionar|interesse|errado'
        ) AS nao_acionar
    FROM cte_tags
)

SELECT 
    phone_number
    --fail,
    --nao_acionar
FROM cte2
WHERE 
    nao_acionar = TRUE
    OR fail >= 7
ORDER BY data_tag1 DESC;


"""

#### Atualiza a consulta dos arquivos do limpador #### 

queries ={
    'sispag':qry_sispag,
    'qtd_calls':qtd_calls,
    'tab_olos':tab_olos,
    'tab_blip':tab_blip,
    'qry_leadscore_olos':lead_score_olos,
    'qry_leadscore_blip':lead_score_blip
}

dfs = {}

for nome, qry in queries.items():
    dfs[nome] = executar_qry(qry,nome)

blacklist = Path(r'C:\bases\Limpador\black_list_new.xlsx')    
df_blacklist = pd.read_excel(blacklist,dtype=str)





Executando: sispag...
✅ sispag: 2933 linhas | ⏱️4.17s
Executando: qtd_calls...
✅ qtd_calls: 270689 linhas | ⏱️9.11s
Executando: tab_olos...
✅ tab_olos: 59997 linhas | ⏱️6.45s
Executando: tab_blip...
✅ tab_blip: 224064 linhas | ⏱️24.20s
Executando: qry_leadscore_olos...
✅ qry_leadscore_olos: 28127 linhas | ⏱️36.77s
Executando: qry_leadscore_blip...
✅ qry_leadscore_blip: 4045 linhas | ⏱️1.90s


In [48]:
pasta = r"M:\Comercial\Call Center - OLOS\01.MAILINGS IMPORTADOS\SECAD\RANGE_VALIDACAO\upload"
lista_dfs = []
for arquivo in Path(pasta).glob('*'):
    if arquivo.name.startswith('.') or arquivo.name.startswith('~'):
        continue

    try:
        if arquivo.suffix.lower() in ['.csv','.txt']:
            df = pd.read_csv(arquivo,encoding='latin-1',sep=None,engine='python',on_bad_lines='skip')
        elif arquivo.suffix.lower() in ['.xlsx','.xls']:
            df = pd.read_excel(arquivo)
        else:
            continue

        df['Nome da Origem'] = arquivo.name

        lista_dfs.append(df)
    except Exception as e:
        print(f'Erro ao ler {arquivo.name}: {e}')
        continue

try:
    rodando_atualmente = pd.concat(lista_dfs,ignore_index=True)
    colunas_remover = [
        'Id_do_cliente_', 'Nome', 'CPF', 'Fone_4', 'ORIGEM', 
        'ORIGEM_DO_LEAD2', 'PROFISSAO', 'ESPECIALIDADE', 'LEAD_SCORING', 
        'PRODUTO', 'A_LTIMO_REGISTRO', 'LOGRADOURO', 'BAIRRO', 
        'CIDADE', 'UF', 'CEP', 'DATA_NASCIMENTO', 'DADOS_DO_PAGAMENTO', 
        'MOTIVO_DE_CANCELAMENTO', 'URL_2', 'Fone_3'
    ]

    df_rod_atual = rodando_atualmente.drop(columns=colunas_remover,errors='ignore')
    df_rod_atual['Fone_1'] = df_rod_atual['Fone_1'].astype(str)
    df_rod_atual['Fone_1'] = df_rod_atual['Fone_1'].str.replace(r'\D','',regex=True)
    df_rod_atual = df_rod_atual.drop_duplicates(subset=['Fone_1'],keep='first')
    df_rod_atual = df_rod_atual.dropna(how='all')
    df_rod_atual.columns = df_rod_atual.columns.str.lower()
    df_rod_atual = df_rod_atual.reset_index(drop=True)
except Exception as e:
    print(f'Erro ao processar base de rodando atualmente: {e}')
    df_rod_atual = None    

Base Crua: Builder
Oriem:
    - Arquivo de base a serem tratadas e posteriormente acionadas
Filtros:
- Area
- Program
- Status Cancelado ou Inativo (foco em base cancelado e troca de ciclo)


In [49]:
### IMPORTAÇÃO E LEITURA DE BASE CRUA ####

Caminho_builder = Path(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\Construtores\base_builder_V2.xlsm')
sheets_builder = [
    'base_carrinho',
    'base_inativa',
    'base_ATIVA'
]

bases = {}

for sheet in sheets_builder:
    df = pd.read_excel(
        Caminho_builder,
        sheet_name=sheet,
        dtype=str,
        engine='openpyxl'
    )

    bases[sheet] = df
    print(f'Sheet "{sheet}" carregada: {df.shape}')

df_inativa = bases['base_inativa']
df_ativa = bases['base_ATIVA']
df_carrinho = bases['base_carrinho']

def diagnostico_copy(df,nome):
    print(f'\n📊 {nome}')
    print('Linhas totais        :', df.shape[0])
    print('Copy nulos           :', df['copy'].isna().sum())
    print('Copy vazios ("")     :', (df['copy'].str.strip() == '').sum())
    print('Copy duplicados      :', df['copy'].duplicated().sum())

diagnostico_copy(df_inativa, 'Base Inativa')
diagnostico_copy(df_ativa, 'Base Ativa')
diagnostico_copy(df_carrinho, 'Base Carrinho')    

df_inativa = bases['base_inativa']
df_inativa = df_inativa.sort_values(by=['copy'],ascending=True)
df_inativa = df_inativa.drop_duplicates(subset='copy',keep='first')
df_inativa = df_inativa.reset_index(drop=True)

df_ativa = bases['base_ATIVA']
df_ativa = df_ativa.sort_values(by=['copy'], ascending=True)
df_ativa = df_ativa.drop_duplicates(subset='copy',keep='first')
df_ativa = df_ativa.reset_index(drop=True)

df_carrinho = bases['base_carrinho']
df_carrinho = df_carrinho.sort_values(by=['copy'],ascending=True)
df_carrinho = df_carrinho.drop_duplicates(subset='copy',keep='first')
df_carrinho = df_carrinho.reset_index(drop=True)
    

Sheet "base_carrinho" carregada: (36837, 12)
Sheet "base_inativa" carregada: (132008, 18)
Sheet "base_ATIVA" carregada: (25113, 25)

📊 Base Inativa
Linhas totais        : 132008
Copy nulos           : 0
Copy vazios ("")     : 0
Copy duplicados      : 32169

📊 Base Ativa
Linhas totais        : 25113
Copy nulos           : 0
Copy vazios ("")     : 0
Copy duplicados      : 3868

📊 Base Carrinho
Linhas totais        : 36837
Copy nulos           : 0
Copy vazios ("")     : 0
Copy duplicados      : 819


In [50]:
def normalizar(txt):
    return (
        unicodedata.normalize('NFKD', txt)
        .encode('ascii', errors='ignore')
        .decode('utf-8')
        .upper()
    )

p = df_carrinho['programa'].astype(str).apply(normalizar)

condicoes = [
    # mais específicas primeiro
    p.str.contains(r'PRONEUROPSI|PROAPSI|PROCOGNITIVA|PROPSICO'),
    p.str.contains(r'PROPSICOMED|PROPSIQ'),
    p.str.contains(r'PROEMPED|PRONEUROPED|PROPED|PRORN|PROTIPED'),
    p.str.contains(r'PROENF/APS|PROENF/SMN|PROENF/TI|PROENF-URG'),
    p.str.contains(r'PROFISIO/TRAUMA|PROFISIO/TO\+|PROFISIO/TIA|PROFISIO/PED|PROFISIO/NEURO|PROFISIO/ESP|PROFISIO/CARDIO'),
    p.str.contains(r'PROPALIATIVO'),
    p.str.contains(r'PRONUTRI'),
    p.str.contains(r'PROMEVET'),
    p.str.contains(r'PROACI|PROAGO|PROAMI|PROANESTESIA|PROATO|PROCARDIOL|PROCLIM|PROENDOCRINO|PROENDOGASTRO|PROGER|PROMEDE|PROMEF|PRONEURO|PRO-ORL|PRORAD|PROTERAPEUTICA|PROURGEM')
]

valores = [
    'Psicologia',
    'Psiquiatria',
    'Pediatria',
    'Enfermagem',
    'Fisioterapia',
    'Multi',
    'Nutrição',
    'Veterinária',
    'Medicina'
]

df_carrinho['Area'] = np.select(condicoes, valores, default='Não identificado')


-- COLA DE COMO CHAMAR A FUNÇÃO A SER TRATADA -- 

nome_base = padrao_e_filtro(
    df_inativa,  ## <- atenção aqui - OBRIGATORIO
    DePara_colunas['TIPO_BASE'], - OBRIGATORIO
    dfs['sispag'], - Não Alterar OBRIGATORIO
    area = None, - Opcional
    type='', - OBRIGATORIO
    program=None, - Opcional
    df_blacklist = df_blacklist, - Não alterar OBRIGATORIO
    leadScoreOlos = dfs['qry_leadscore_olos'], - Não alterar OBRIGATORIO
    leadScoreBlip = dfs['qry_leadscore_blip'], - Não alterar OBRIGATORIO
    qtdcalls = dfs['qtd_calls'], - Não alterar OBRIGATORIO
    limite=10, - Opcional
    olos_ultimo_contato=dfs['tab_olos'], - Não alterar OBRIGATORIO
    blip_ultimo_contato=dfs['tab_blip'] - Não alterar OBRIGATORIO
)

⚠️ Observação:
- Para não aplicar filtros, utilize `None`
- Não utilizar strings vazias ('') para área ou programa
- DePara_colunas utilizar ->  Base_ativa, Base_inativa, Base_carrinho
- area utilizar area especifica de atuação -> Fisioterapia, Enfermagem, Psicologia(se Inativa) Saúde Mental(se Ativa), Medicina, Nutrição, Veterinaria
- type Alterar se a base for INATIVA, utilizar -> Inativo, Cancelado
- Program Preencher com o programa de interesse
- limite -> quantidade limite de calls que o contato recebeu, valor padrão é 10
- Campos marcados como NÃO ALTERAR, não sofrem modificações de valores e/ou parametros
